# RAG-Based Domain-Specific AI Assistant
**Domain:** College / Student Information
**Documents:** Fee Structure, Attendance & Academic Policy, Academic Calendar & Examination Rules
**Author:** Sameer Shaik

This notebook implements the full RAG pipeline required by the assignment:
`documents -> text extraction -> chunking -> embeddings -> similarity retrieval -> context -> LLM generation (optional) -> grounded answer`


## 1. Setup — install required libraries
Per the *Supported Libraries* document, the minimal required stack is:
`pymupdf + sentence-transformers + scikit-learn + numpy` (+ `openai`, optional).

In [10]:
!pip install -q pymupdf sentence-transformers scikit-learn numpy openai

## 2. Upload the three source PDFs
Upload `Sample_PDF_1_College_Fee_Structure.pdf`, `Sample_PDF_2_Attendance_Academic_Policy.pdf`, and `Sample_PDF_3_Academic_Calendar_Exam_Rules.pdf` into a `data/` folder.

In [11]:
import os
os.makedirs("data", exist_ok=True)

from google.colab import files
uploaded = files.upload()  # select the 3 sample PDFs
for fname in uploaded:
    os.rename(fname, os.path.join("data", fname))
print(os.listdir("data"))

Saving Sample_PDF_3_Academic_Calendar_Exam_Rules_260814_172606.pdf to Sample_PDF_3_Academic_Calendar_Exam_Rules_260814_172606.pdf
Saving Sample_PDF_2_Attendance_Academic_Policy_260814_172525.pdf to Sample_PDF_2_Attendance_Academic_Policy_260814_172525.pdf
Saving Sample_PDF_1_College_Fee_Structure_260814_172341.pdf to Sample_PDF_1_College_Fee_Structure_260814_172341.pdf
['Sample_PDF_2_Attendance_Academic_Policy_260814_172525.pdf', 'Sample_PDF_3_Academic_Calendar_Exam_Rules_260814_172606.pdf', 'Sample_PDF_1_College_Fee_Structure_260814_172341.pdf']


## 3. Imports and configuration

In [30]:
import glob
import json
import numpy as np
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = "data"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
CHUNK_WORD_SIZE = 180      # words per chunk
CHUNK_OVERLAP = 30         # word overlap between consecutive chunksi
TOP_K = 3                  # chunks retrieved per question
NOT_FOUND_THRESHOLD = 0.35 # below this cosine similarity -> "not found"

# Set this if you want real LLM-based generation (Step 11).
# Leaving it unset is fine -- the pipeline falls back to returning the
# top retrieved chunk as the grounded answer, which still satisfies the
# assignment's core retrieval requirement (see Supported Libraries doc).
os.environ.setdefault("OPENAI_API_KEY", "")
USE_LLM_GENERATION = bool(os.environ.get("OPENAI_API_KEY"))

## Step 1 & 2 — Load PDFs and extract text (with page metadata)

In [32]:
import re


def clean_extracted_text(text):
    text = re.sub(
        r'(?<![A-Za-z])I(?=\d{1,3}(?:,\d{3})+)',
        '₹',
        text
    )
    return text


def load_and_extract(data_dir):
    records = []

    for path in sorted(glob.glob(os.path.join(data_dir, "*.pdf"))):
        filename = os.path.basename(path)

        doc = fitz.open(path)

        for page_index in range(len(doc)):
            text = doc.load_page(page_index).get_text("text").strip()

            # Fix incorrectly extracted ₹ symbols
            text = clean_extracted_text(text)

            if text:
                records.append({
                    "source": filename,
                    "page": page_index + 1,
                    "text": text
                })

        doc.close()

    return records


page_records = load_and_extract(DATA_DIR)

print(
    f"Extracted {len(page_records)} page(s) "
    f"from {len(set(r['source'] for r in page_records))} document(s)."
)

Extracted 3 page(s) from 3 document(s).


## Step 3 & 4 — Chunk text and attach metadata
**Chunking strategy: 180-word chunks with 30-word overlap.**
These policy-style PDFs are short (~300-500 words/page). A 180-word chunk keeps a full logical
section (e.g. one attendance rule) intact instead of splitting it mid-idea, while the 30-word
overlap prevents a fact that sits on a chunk boundary from being separated during retrieval.

## Step 5 & 6 — Create chunk embeddings and store them (NumPy array = our in-memory vector store)

In [33]:
model = SentenceTransformer(EMBEDDING_MODEL_NAME)
texts = [c["text"] for c in chunks]
chunk_embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
print("Embedding matrix shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (6, 384)


## Step 7, 8, 9 — Embed question, calculate similarity, select top-k chunks

In [35]:
def retrieve_top_k(question, k=TOP_K):
    q_emb = model.encode([question], convert_to_numpy=True)
    sims = cosine_similarity(q_emb, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    return [{"chunk": chunks[i], "score": float(sims[i])} for i in top_idx]

## Step 10 — Build context from retrieved chunks

In [36]:
def build_context(retrieved):
    parts = [f"[Source: {r['chunk']['source']}, Page {r['chunk']['page']}]\n{r['chunk']['text']}" for r in retrieved]
    return "\n\n".join(parts)

## Step 11 — Ask LLM to answer from context (OPTIONAL)
Per the *Supported Libraries* doc, LLM generation is optional for the basic retrieval demonstration. If no `OPENAI_API_KEY` is set, we fall back to returning the top retrieved chunk as the grounded, extractive answer.

In [37]:
from openai import OpenAI

def generate_llm_answer(question, context):
    client = OpenAI()
    system_prompt = (
        "You are a college assistant. Answer the question using ONLY the "
        "provided context. If the answer is not present in the context, reply "
        "exactly: 'Answer not found in the provided documents.' Be concise."
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}],
        temperature=0
    )
    return resp.choices[0].message.content.strip()

def generate_extractive_answer(retrieved, best_score):
    if best_score < NOT_FOUND_THRESHOLD:
        return "Answer not found in the provided documents."
    return retrieved[0]["chunk"]["text"]

In [39]:
def chunk_text(page_records, chunk_size=CHUNK_WORD_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    chunk_id = 0
    for record in page_records:
        words = record["text"].split()
        start = 0
        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunks.append({
                "chunk_id": chunk_id,
                "source": record["source"],
                "page": record["page"],
                "text": " ".join(words[start:end])
            })
            chunk_id += 1
            if end == len(words):
                break
            start = end - overlap
    return chunks

chunks = chunk_text(page_records)
print(f"Created {len(chunks)} chunks.")
chunks[0]

Created 6 chunks.


{'chunk_id': 0,
 'source': 'Sample_PDF_1_College_Fee_Structure_260814_172341.pdf',
 'page': 1,
 'text': 'Page 1 Sample PDF 1 — College Fee Structure Practice document for the Generative AI RAG project Institution ABC Institute of Technology Undergraduate Programs B.Tech — Computer Science: ₹80,000 per semester. B.Tech — Electronics: ₹72,000 per semester. B.Tech — Mechanical: ₹68,000 per semester. Postgraduate Programs M.Tech — Artificial Intelligence: ₹95,000 per semester. MBA: ₹75,000 per semester. Additional Charges Admission processing fee: ₹2,000 one time. Examination fee: ₹1,500 per semester. Library fee: ₹1,000 per semester. Technology fee: ₹2,500 per semester. Hostel Charges Standard hostel: ₹45,000 per semester. Premium hostel: ₹60,000 per semester. Hostel charges do not include the refundable security deposit of ₹5,000. Payment Schedule At least 50% of the semester tuition fee must be paid before the semester begins. The remaining amount must be paid by the end of the first mo

## Step 12 — Full QA wrapper: display concise answer + source

In [40]:
def answer_question(question, k=TOP_K, verbose=True):
    retrieved = retrieve_top_k(question, k=k)
    best_score = retrieved[0]["score"]
    context = build_context(retrieved)

    if best_score < NOT_FOUND_THRESHOLD:
        answer, used_llm = "Answer not found in the provided documents.", False
    elif USE_LLM_GENERATION:
        try:
            answer, used_llm = generate_llm_answer(question, context), True
        except Exception:
            answer, used_llm = generate_extractive_answer(retrieved, best_score), False
    else:
        answer, used_llm = generate_extractive_answer(retrieved, best_score), False

    result = {
        "question": question, "answer": answer, "top_score": round(best_score, 4),
        "used_llm": used_llm,
        "sources": [{"source": r["chunk"]["source"], "page": r["chunk"]["page"],
                     "score": round(r["score"], 4), "chunk_id": r["chunk"]["chunk_id"]} for r in retrieved]
    }
    if verbose:
        print(f"\nQ: {question}\nA: {answer}")
        print(f"   [top similarity: {result['top_score']} | generation: {'LLM' if used_llm else 'extractive/fallback'}]")
        for s in result["sources"]:
            print(f"   -> {s['source']} (page {s['page']}, chunk {s['chunk_id']}, score {s['score']})")
    return result

## Demo — 8 required test questions
- **5 answerable** (direct wording match)
- **2 deliberately unanswerable** (grounding / "not found" test)
- **1 semantic-search example** (question wording differs from the document wording)

In [41]:
DEMO_QUESTIONS = [
    "What is the minimum attendance percentage required for exams?",
    "What is the tuition fee for B.Tech Computer Science per semester?",
    "When does the Semester I end-semester examination take place?",
    "What is the refund percentage if admission is cancelled before the semester begins?",
    "What grade is awarded for marks between 70 and 79?",
    "What is the hostel curfew time?",
    "Is there a fee waiver for sports achievements?",
    "How many classes can I skip and still be allowed to sit for exams?",
]

results = [answer_question(q) for q in DEMO_QUESTIONS]

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved results.json")


Q: What is the minimum attendance percentage required for exams?
A: Page 1 Sample PDF 2 — Attendance & Academic Policy Practice document for the Generative AI RAG project Minimum Attendance Students must maintain a minimum of 75% attendance in each course to be eligible for the regular end-semester examination. Attendance is calculated separately for each course. Attendance Between 65% and 74% A student with attendance between 65% and 74% may apply for attendance condonation when supported by an approved reason. The maximum condonation permitted is 10 percentage points, subject to institutional approval. Attendance Below 65% Students with attendance below 65% are normally not eligible for condonation and may be detained from the regular examination. Medical Leave Medical leave must be supported by appropriate medical documentation. Submission of medical documents does not automatically guarantee approval. Attendance Calculation Attendance percentage = classes attended ÷ classes conduc

## Results table

In [43]:
import pandas as pd

rows = []
for r in results:
    top = r["sources"][0]
    rows.append({
        "Question": r["question"],
        "Top Source": f"{top['source']} (p.{top['page']})",
        "Similarity": r["top_score"],
        "Generation": "LLM" if r["used_llm"] else "Extractive/Fallback",
        "Answer (truncated)": (r["answer"][:80] + "...") if len(r["answer"]) > 80 else r["answer"]
    })

pd.DataFrame(rows)

,Question,Top Source,Similarity,Generation,Answer (truncated)
0,What is the minimum attendance percentage requ...,Sample_PDF_2_Attendance_Academic_Policy_260814...,0.6072,Extractive/Fallback,Page 1 Sample PDF 2 — Attendance & Academic Po...
1,What is the tuition fee for B.Tech Computer Sc...,Sample_PDF_1_College_Fee_Structure_260814_1723...,0.6846,Extractive/Fallback,Page 1 Sample PDF 1 — College Fee Structure Pr...
2,When does the Semester I end-semester examinat...,Sample_PDF_3_Academic_Calendar_Exam_Rules_2608...,0.6895,Extractive/Fallback,Page 1 Sample PDF 3 — Academic Calendar & Exam...
3,What is the refund percentage if admission is ...,Sample_PDF_1_College_Fee_Structure_260814_1723...,0.6590,Extractive/Fallback,"cancelled before the semester begins, 80% of t..."
4,What grade is awarded for marks between 70 and...,Sample_PDF_3_Academic_Calendar_Exam_Rules_2608...,0.3666,Extractive/Fallback,Page 1 Sample PDF 3 — Academic Calendar & Exam...
5,What is the hostel curfew time?,Sample_PDF_3_Academic_Calendar_Exam_Rules_2608...,0.2664,Extractive/Fallback,Answer not found in the provided documents.
6,Is there a fee waiver for sports achievements?,Sample_PDF_1_College_Fee_Structure_260814_1723...,0.3357,Extractive/Fallback,Answer not found in the provided documents.
7,How many classes can I skip and still be allow...,Sample_PDF_2_Attendance_Academic_Policy_260814...,0.4885,Extractive/Fallback,Page 1 Sample PDF 2 — Attendance & Academic Po...


## Notes on Responsible AI (see project report for full discussion)
- **Grounding / hallucination control:** answers below `NOT_FOUND_THRESHOLD` return an explicit "not found" message instead of inventing content.
- **Traceability:** every answer is returned with its source document, page number, and chunk id.
- **Privacy:** only the 3 sample institutional PDFs are used; no personal student data is processed.
- **Human-in-the-loop:** low-confidence or "not found" answers should route to a human (e.g. academic office) rather than being treated as final.
- **Bias/fairness & auditing:** retrieval is scored purely on embedding similarity to the source text, not demographic attributes; `results.json` provides an auditable log of every question, retrieved chunk, and score.